# What is in Tulika's file that the new GTEx MutPred2 file is missing?

Tulika's scores are used here **only as a reference for what is missing** — to find the gaps in
our own score set. None of her scores are read as values, and nothing here proposes filling a
gap from her files.

The comparison follows three steps, in this order:

1. take the variants in `extractedVariants.xlsx`
2. of those, keep the ones Tulika has a MutPred2 score for
3. of those, check which are present in the new `MutPred2_gtex.tsv` — and which are missing

Restricting to the workbook first matters: Tulika's table also covers variants outside the
current extraction, and those are not gaps in our file, just variants we are not analysing.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, ExtractedVariants_file, MutPred2_gtex_file, GTeX_file

DATA        = Path(str(DATA_DIR))
TULIKA_FILE = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"
OUT_DIR     = DATA / "AI_generated"

SHEETS = ["CM", "PCD", "PH", "PKD", "ASD", "nonTarget"]

for f in (ExtractedVariants_file, MutPred2_gtex_file, TULIKA_FILE):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/extractedVariants.xlsx
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/MutPred2_gtex.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/mutpred_score_variant_transcript_tulika.tsv


## 2. Step 1 — the workbook variants

In [2]:
xl = pd.ExcelFile(ExtractedVariants_file)
sheets = {name: xl.parse(name) for name in SHEETS}

workbook = set()
for name in SHEETS:
    workbook |= set(sheets[name]["Variant"].dropna().unique())

print(f"workbook variants (target + nonTarget) : {len(workbook)}")

workbook variants (target + nonTarget) : 60183


## 3. Step 2 — which of them Tulika has a score for

Her table is one row per variant x transcript (`mutation` = the genomic variant, `Feature` = the
versioned transcript, `mutpred` = the score). Rows with no score are dropped before anything else.

In [3]:
tulika = pd.read_csv(TULIKA_FILE, sep="\t", low_memory=False)
tulika = tulika[tulika["mutpred"].notna()]

TULIKA_ALL = set(tulika["mutation"].dropna().unique())
in_tulika  = workbook & TULIKA_ALL

print(f"Tulika's file        : {len(tulika)} scored rows, {len(TULIKA_ALL)} distinct variants")
print(f"  outside the workbook (ignored from here on) : {len(TULIKA_ALL - workbook)}")
print()
print(f"workbook variants she has a score for : {len(in_tulika)} "
      f"({len(in_tulika)/len(workbook):.1%} of the workbook)")

Tulika's file        : 632653 scored rows, 96253 distinct variants
  outside the workbook (ignored from here on) : 59341

workbook variants she has a score for : 36912 (61.3% of the workbook)


## 4. Step 3 — how many of those are in the new file

This is the answer to the question. `MutPred2_gtex.tsv` is the file built from SJ's run plus
Tim's run; a variant counts as present if it has at least one scored transcript row.

In [4]:
new = pd.read_csv(MutPred2_gtex_file, sep="\t")
NEW = set(new["Variant"])

present = in_tulika & NEW
missing = in_tulika - NEW

print(f"new file : {len(new)} rows, {len(NEW)} distinct variants\n")
print(f"workbook variants Tulika has          : {len(in_tulika):>6}")
print(f"  ALSO in the new GTEx file           : {len(present):>6}  ({len(present)/len(in_tulika):.2%})")
print(f"  MISSING from the new GTEx file      : {len(missing):>6}  ({len(missing)/len(in_tulika):.2%})")

new file : 164280 rows, 50323 distinct variants

workbook variants Tulika has          :  36912
  ALSO in the new GTEx file           :  36526  (98.95%)
  MISSING from the new GTEx file      :    386  (1.05%)


### 4a. Per sheet

Where the missing variants sit. `tulika` is the step-2 set for that sheet; `missing` is how many
of those the new file does not cover.

In [5]:
rows = []
for name in SHEETS:
    v = set(sheets[name]["Variant"].dropna().unique())
    a = v & TULIKA_ALL
    rows.append({"sheet": name, "workbook": len(v), "tulika": len(a),
                 "in_new": len(a & NEW), "missing": len(a - NEW)})

cmp = pd.DataFrame(rows)
cmp["pct_missing"] = (100 * cmp["missing"] / cmp["tulika"]).round(2)
cmp.loc[len(cmp)] = {"sheet": "ALL (union)", "workbook": len(workbook), "tulika": len(in_tulika),
                     "in_new": len(present), "missing": len(missing),
                     "pct_missing": round(100 * len(missing) / len(in_tulika), 2)}
print(cmp.to_string(index=False))

      sheet  workbook  tulika  in_new  missing  pct_missing
         CM     27540   25409   25049      360         1.42
        PCD      8828    7009    6999       10         0.14
         PH       414     342     339        3         0.88
        PKD      2263    1721    1716        5         0.29
        ASD      1425     498     495        3         0.60
  nonTarget     19718    1938    1933        5         0.26
ALL (union)     60183   36912   36526      386         1.05


### 4b. Export the missing variants

In [6]:
miss_df = (tulika[tulika["mutation"].isin(missing)]
             .groupby("mutation")
             .agg(gene=("external_gene_name", "first"),
                  n_transcripts=("Feature", "nunique"),
                  max_mutpred=("mutpred", "max"))
             .reset_index()
             .rename(columns={"mutation": "Variant"}))

sheet_of = {}
for name in SHEETS:
    for v in sheets[name]["Variant"].dropna().unique():
        sheet_of.setdefault(v, []).append(name)
miss_df["sheets"] = miss_df["Variant"].map(lambda v: ",".join(sheet_of.get(v, [])))

out = OUT_DIR / "in_tulika_missing_from_gtex.tsv"
miss_df.to_csv(out, sep="\t", index=False)

print(f"{len(miss_df)} variants written to\n  {out}\n")
print(miss_df["sheets"].value_counts().head(10).to_string())
print()
miss_df.head(10)

386 variants written to
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/in_tulika_missing_from_gtex.tsv

sheets
CM           360
PCD           10
nonTarget      5
PKD            5
ASD            3
PH             3



,Variant,gene,n_transcripts,max_mutpred,sheets
0,10_110820072_G/A,RBM20,1,0.034,CM
1,10_119651854_A/G,BAG3,1,0.142,CM
2,10_119669851_G/A,BAG3,2,0.149,CM
3,10_119670177_G/C,BAG3,2,0.072,CM
4,10_119672255_C/T,BAG3,2,0.083,CM
5,10_119672654_C/G,BAG3,2,0.238,CM
6,10_20808510_C/G,NEBL,1,0.564,CM
7,10_20817691_A/G,NEBL,2,0.186,CM
8,10_20819425_G/A,NEBL,3,0.062,CM
9,10_20823209_T/A,NEBL,2,0.203,CM


## 5. Secondary — the same comparison at transcript level

The variant-level answer above is the one that was asked for. This section exists as a warning:
joining the two files on (variant, **transcript**) gives a far worse-looking match rate, and
almost all of that is a transcript-versioning artefact rather than missing scores.

In [7]:
tw = tulika[tulika["mutation"].isin(in_tulika)]

GTEX     = set(pd.read_csv(GTeX_file, sep="\t", usecols=["transcript_id"])["transcript_id"])
GTEX_IDS = {t.split(".")[0] for t in GTEX}

pairs = (pd.DataFrame({"Variant": tw["mutation"], "ENST": tw["Feature"]})
           .drop_duplicates())
pairs["ENST_nov"] = pairs["ENST"].str.split(".").str[0]

new_keys     = set(zip(new["Variant"], new["ENST"]))
new_keys_nov = set(zip(new["Variant"], new["ENST"].str.split(".").str[0]))

pairs["exact"]         = [k in new_keys     for k in zip(pairs["Variant"], pairs["ENST"])]
pairs["ignoring_ver"]  = [k in new_keys_nov for k in zip(pairs["Variant"], pairs["ENST_nov"])]
pairs["in_gtex_exact"] = pairs["ENST"].isin(GTEX)
pairs["in_gtex_id"]    = pairs["ENST_nov"].isin(GTEX_IDS)

print(f"Tulika (variant, transcript) pairs for these variants : {len(pairs)}")
print(f"  matched on exact ENST version                       : {pairs['exact'].sum():>7}  ({pairs['exact'].mean():.1%})")
print(f"  matched ignoring the version suffix                 : {pairs['ignoring_ver'].sum():>7}  ({pairs['ignoring_ver'].mean():.1%})")

Tulika (variant, transcript) pairs for these variants : 119771
  matched on exact ENST version                       :   44712  (37.3%)
  matched ignoring the version suffix                 :   88468  (73.9%)


### 5a. Why the unmatched pairs are unmatched

In [8]:
ab = pairs[~pairs["exact"]]

not_gtex   = (~ab["in_gtex_id"]).sum()
wrong_ver  = (ab["in_gtex_id"] & ~ab["in_gtex_exact"]).sum()
recovered  = (ab["in_gtex_id"] & ~ab["in_gtex_exact"] & ab["ignoring_ver"]).sum()
genuine    = ab["in_gtex_exact"].sum()

print(f"unmatched pairs : {len(ab)}\n")
print(f"  transcript ID not in the GTEx set at all          : {not_gtex:>7}")
print(f"  ID in GTEx, but Tulika carries another version    : {wrong_ver:>7}")
print(f"    of which the new file DOES score that variant")
print(f"    on the same transcript ID                       : {recovered:>7}")
print(f"  version fine, pair genuinely not scored by us     : {genuine:>7}")
print()
print("So the transcript-level gap is dominated by version differences, not by absent scores -")
print("which is the same reason the BrainSpan file was kept out of the GTEx build.")

unmatched pairs : 75059

  transcript ID not in the GTEx set at all          :    5111
  ID in GTEx, but Tulika carries another version    :   68960
    of which the new file DOES score that variant
    on the same transcript ID                       :   43756
  version fine, pair genuinely not scored by us     :     988

So the transcript-level gap is dominated by version differences, not by absent scores -
which is the same reason the BrainSpan file was kept out of the GTEx build.


## 6. Summary

In [9]:
print("Excel workbook -> Tulika -> new GTEx file")
print(f"  {len(workbook)} workbook variants")
print(f"  {len(in_tulika)} of them have a Tulika score")
print(f"  {len(present)} of those are in MutPred2_gtex.tsv ({len(present)/len(in_tulika):.2%})")
print(f"  {len(missing)} are MISSING -> {(OUT_DIR / 'in_tulika_missing_from_gtex.tsv').name}")

Excel workbook -> Tulika -> new GTEx file
  60183 workbook variants
  36912 of them have a Tulika score
  36526 of those are in MutPred2_gtex.tsv (98.95%)
  386 are MISSING -> in_tulika_missing_from_gtex.tsv
